### Stage 1 — Simple batch: land a bronze table
Ingestion = fetch → flatten → **append** to a bronze table in Unity Catalog.
Bronze keeps raw snapshots: every run appends the full 7-day window with a `fetched_at` timestamp
(the same hour will appear in many snapshots with different values as forecasts get revised).
Reconciliation — latest value per hour — is a silver-layer job later.

In [0]:
# Ingest: append this forecast snapshot to the bronze table
import requests

latitude = float(dbutils.widgets.get("latitude"))
longitude = float(dbutils.widgets.get("longitude"))

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,precipitation",
    },
)
response.raise_for_status()

spark.createDataFrame([(response.text,)], "payload STRING").createOrReplaceTempView("open_meteo_raw")

target_table = "workspace.open_meteo.bronze_hourly_forecast"

bronze_df = spark.sql("""
  WITH parsed AS (
    SELECT from_json(
      payload,
      'STRUCT<
        latitude: DOUBLE,
        longitude: DOUBLE,
        elevation: DOUBLE,
        hourly: STRUCT<
          time: ARRAY<TIMESTAMP>,
          temperature_2m: ARRAY<DOUBLE>,
          precipitation: ARRAY<DOUBLE>
        >
      >'
    ) AS d
    FROM open_meteo_raw
  )
  SELECT
    current_timestamp() AS fetched_at,
    d.latitude,
    d.longitude,
    d.elevation,
    h.time,
    h.temperature_2m,
    h.precipitation
  FROM parsed,
  LATERAL inline_outer(
    arrays_zip(d.hourly.time, d.hourly.temperature_2m, d.hourly.precipitation)
  ) AS h(time, temperature_2m, precipitation)
""")

bronze_df.write.mode("append").saveAsTable(target_table)

In [0]:
# Verify: one snapshot per fetch, 168 hourly rows per snapshot (7 days x 24 h)
display(spark.sql(f"""
  SELECT
    count(*)                   AS total_rows,
    count(DISTINCT fetched_at) AS snapshots,
    min(time)                 AS earliest_hour,
    max(time)                  AS latest_hour
  FROM {target_table}
"""))

In [0]:
%sql
select * from workspace.open_meteo.bronze_hourly_forecast